# Cyclic peptide binders: designing a macrocycle against a protein target


A cyclic peptide binder is designed *de novo* against a target that keeps its
experimental coordinates and hotspots. This tutorial builds one for each of the three
cyclization chemistries, generates designs, and checks two things that are easy to
conflate: whether the ring closed, and whether it landed where it was asked to.

| Chemistry | Closing bond | Ideal |
|---|---|---|
| Head-to-tail | C-terminal `C` to N-terminal `N` (amide) | 1.33 A |
| Disulfide | `CYS.SG` to `CYS.SG` | 2.05 A |
| Isopeptide | `LYS.NZ` to `ASP.CG` (amide) | 1.33 A |

**The regime matters more than the chemistry.** When the ring has input coordinates, all
three chemistries close reliably once the crosslinked residues are atomized (see the
[cyclic peptide tutorial](cyclic_peptide_tutorial.ipynb)). A *designed* binder has no input
coordinates, so atomization is unavailable: every atomized token must carry a finite
coordinate at inference. That rules out the fix exactly where the crosslinks need it, and
the results below follow from this.

Target is PD-1 (`01_PD1` from the PPI benchmark), hotspots VAL33, ILE95, LEU97, GLN102.

## Setup

In [ ]:
from pathlib import Path
import os, subprocess, glob

import numpy as np

REPO_ROOT = Path(os.environ.get("RFD4_REPO", "")).resolve() if os.environ.get("RFD4_REPO") else None
if REPO_ROOT is None:
    here = Path.cwd()
    for cand in (here, *here.parents):
        if (cand / ".project-root").exists():
            REPO_ROOT = cand
            break
assert REPO_ROOT is not None, "Run inside the repository, or set RFD4_REPO."

WORK = Path(os.environ.get("CYCLIC_BINDER_WORK", REPO_ROOT / "scratch" / "cyclic_binder_tutorial"))
WORK.mkdir(parents=True, exist_ok=True)

INPUTS = REPO_ROOT / "docs" / "examples" / "inputs" / "cyclic_binders"
CHEMISTRIES = ["head_tail", "disulfide", "isopeptide"]
HOTSPOTS = [33, 95, 97, 102]          # PD-1 target residues, as annotated in the benchmark
LENGTH = 12                            # ring size; the disulfide's useful band is 10-16

print("repo:", REPO_ROOT)
print("inputs:", [p.name for p in sorted(INPUTS.glob("*.cif.zst"))])

## 1. What the input contains

Each input is two chains: the **target** (chain `B`), with its coordinates, full sequence
condition and hotspots preserved from the benchmark, and the **peptide** (chain `P`), which
has no coordinate conditioning at all — its structure and sequence are both generated. Only
the two residues the chemistry depends on have their identity fixed.

In [ ]:
import rfproteina.conditions  # noqa: F401 -- registers condition classes; see the traps below
from atomworks.io.parser import parse
from atomworks.io.utils.atom_array_plus import as_atom_array_plus
from rfproteina.utils.parse_presets import ANNOTATED_CIF


def load(chemistry):
    path = INPUTS / f"01_PD1_{chemistry}_{LENGTH:02d}.cif.zst"
    return as_atom_array_plus(parse(str(path), config=ANNOTATED_CIF)["assemblies"]["1"][0])


aa = load("head_tail")
pep, tgt = aa.chain_id == "P", aa.chain_id == "B"
seq = np.asarray(aa.get_annotation("condition_sequence_1_residue"))
hot = np.asarray(aa.get_annotation("condition_hotspot_1_atom")).astype(bool)

print(f"peptide chain P : {int(pep.sum()):4d} atoms, {len(set(aa.res_id[pep].tolist()))} residues")
print(f"target  chain B : {int(tgt.sum()):4d} atoms, {len(set(aa.res_id[tgt].tolist()))} residues")
print(f"peptide sequence condition : {sorted(set(seq[pep].tolist()))}   ('<M>' = designed)")
print(f"hotspot atoms on target    : {int(hot.sum())}")
print(f"bonds (incl. the closure)  : {len(aa.bonds.as_array())}")

### Four ways this input can go silently wrong

Each of these produces **no error** — the run exits `0` and writes a plausible file.

1. **A length-range binder cannot carry a closure bond.** The PPI benchmark specifies
   binders as a placeholder row (`80-150`, one `UNK` atom). A `_struct_conn` record names
   two atoms of two numbered residues, so there has to be something to point at. The
   peptide chain must be enumerated explicitly, which is what
   `docs/examples/scripts/build_cyclic_binders.py` does.
2. **`expseg` bounds default to `-1`, not `0`.** `-1` means "not an expandable segment";
   `0` means "expand to zero residues", and `RealizeExpandableSegments` then deletes the
   chain. Generation still succeeds — with the peptide simply absent from the output.
3. **Two independently parsed chains both number from entity 0** and collide on write, so
   the target ends up referencing the peptide's `_entity_poly_seq`.
4. **`import rfproteina.conditions` is load-bearing.** Without it the condition classes are
   unregistered and the target's hotspots are dropped on write — the same silent-drop
   failure mode as an RCSB `disulf` record, which the parse preset also ignores.

Validate before generating; it catches 1, 3 and the closure itself:

```bash
bash skills/rfd4-proteina-annotated-cif/scripts/check_annotated_cif.sh <input.cif>
```

Look for `struct_conn_ok ... all covale linkages verified in the bond graph`, and for
`struct_conn_bond_missing`, which means the closure never made it into the bond graph.

## 2. The target and its hotspots

The peptide has no coordinates yet, so there is nothing of it to draw. This is the target
it has to bind, with the four hotspots in orange.

In [ ]:
import io
import warnings

import py3Dmol
import biotite.structure.io.pdb as biopdb


def to_pdb(atom_array):
    """Render an AtomArray as a PDB string, which is what the 3Dmol viewers take."""
    f = biopdb.PDBFile()
    with warnings.catch_warnings():
        # biotite's writer trips a numpy DeprecationWarning on every call. Suppressed here
        # rather than globally: importing atomworks resets the filters.
        warnings.simplefilter("ignore", DeprecationWarning)
        f.set_structure(atom_array)
    buf = io.StringIO()
    f.write(buf)
    return buf.getvalue()


target = aa[tgt]
target = target[~np.isnan(target.coord).any(axis=-1)]

view = py3Dmol.view(width=640, height=420)
view.addModel(to_pdb(target), "pdb")
view.setStyle({"cartoon": {"color": "0xb9c0c9"}})
view.addStyle({"resi": HOTSPOTS}, {"stick": {"colorscheme": "orangeCarbon", "radius": 0.3}})
view.setBackgroundColor("0xffffff")
view.zoomTo()
view.show()

## 3. Generate

PPI settings per the [recycling schedule table](../how-to/run-the-model.md#recycling-schedules):
`medium/front_loaded`. Raise `NUM_REPLICATES` for a real campaign; 4 keeps the tutorial short.

In [ ]:
NUM_REPLICATES = int(os.environ.get("CYCLIC_BINDER_REPLICATES", 4))


def generate(chemistry):
    out_dir = WORK / chemistry
    if (out_dir / "inference_samples").exists():
        print(f"{chemistry}: reusing {out_dir}")
        return out_dir
    cmd = [
        "rfd4-generate",
        f"inputs={INPUTS / f'01_PD1_{chemistry}_{LENGTH:02d}.cif.zst'}",
        f"num_replicates={NUM_REPLICATES}",
        f"out_dir={out_dir}",
        "seed=0",
        "batch_size=1",
        "recycle_strategy=medium/front_loaded",
        "compress_outputs=false",
    ]
    print(" ".join(cmd))
    subprocess.run(cmd, cwd=REPO_ROOT, check=True)
    return out_dir


RUNS = {c: generate(c) for c in CHEMISTRIES}

## 4. Did the ring close, and did it bind?

Two separate questions. A ring that closes perfectly in empty space is not a binder, and a
peptide packed against the hotspots with an open ring is not a macrocycle — so both are
measured, and failures stay in the denominator.

The closure test is **two-sided**. A one-sided "closed if under 2.3 A" cut would score an
impossible overshoot as a success: an un-atomized disulfide can land near 1.2 A, well inside
a single sulfur atom.

In [ ]:
import biotite.structure as struc
import biotite.structure.io.pdbx as pdbx

# (atom on residue 1, atom on residue LENGTH, ideal distance, tolerance)
CLOSURE = {
    "head_tail":  ("N",  "C",  1.33, 0.30),
    "disulfide":  ("SG", "SG", 2.05, 0.30),
    "isopeptide": ("NZ", "CG", 1.33, 0.30),
}


def designs(run_dir):
    return sorted(glob.glob(str(Path(run_dir) / "inference_samples" / "**" / "*.cif"), recursive=True))


def measure(path, chemistry):
    arr = pdbx.get_structure(pdbx.CIFFile.read(path), model=1)
    arr = arr[struc.filter_amino_acids(arr)]
    sizes = {c: int((arr.chain_id == c).sum()) for c in np.unique(arr.chain_id)}
    pep_chain, tgt_chain = min(sizes, key=sizes.get), max(sizes, key=sizes.get)
    p, t = arr[arr.chain_id == pep_chain], arr[arr.chain_id == tgt_chain]

    atom_a, atom_b, ideal, tol = CLOSURE[chemistry]

    def coord(sub, res_id, name):
        sel = (sub.res_id == res_id) & (sub.atom_name == name)
        return sub.coord[sel][0] if sel.sum() == 1 else None

    a, b = coord(p, 1, atom_a), coord(p, LENGTH, atom_b)
    d = float(np.linalg.norm(a - b)) if a is not None and b is not None else np.nan

    hs = t[np.isin(t.res_id, HOTSPOTS)]
    dm = np.linalg.norm(p.coord[:, None, :] - hs.coord[None, :, :], axis=-1)
    # Count hotspot *residues* in contact, not their atoms: one residue contributes many atoms.
    engaged = len({int(r) for r, dd in zip(hs.res_id, dm.min(axis=0)) if dd <= 5.0})
    return d, bool(np.isfinite(d) and abs(d - ideal) <= tol), engaged


print(f"{'chemistry':<12}{'closed':>9}{'median':>9}{'range':>16}{'hotspots engaged':>19}")
for c in CHEMISTRIES:
    rows = [measure(p, c) for p in designs(RUNS[c])]
    d = np.array([r[0] for r in rows]); fin = d[np.isfinite(d)]
    n_closed = sum(r[1] for r in rows)
    eng = np.median([r[2] for r in rows])
    rng = f"{fin.min():.2f}-{fin.max():.2f}" if len(fin) else "n/a"
    med = f"{np.median(fin):.3f}" if len(fin) else "n/a"
    print(f"{c:<12}{n_closed:>5}/{len(rows):<3}{med:>9}{rng:>16}{eng:>13.0f} of {len(HOTSPOTS)}")

## 5. What the designs look like

Target in grey, hotspots orange, the macrocycle coloured, and the closure bond itself drawn
as a green dashed line between the two bonded atoms.

In [ ]:
def show(chemistry, index=0):
    path = designs(RUNS[chemistry])[index]
    arr = pdbx.get_structure(pdbx.CIFFile.read(path), model=1)
    arr = arr[struc.filter_amino_acids(arr)]
    sizes = {c: int((arr.chain_id == c).sum()) for c in np.unique(arr.chain_id)}
    pep_chain, tgt_chain = min(sizes, key=sizes.get), max(sizes, key=sizes.get)
    p = arr[arr.chain_id == pep_chain]

    atom_a, atom_b, ideal, _ = CLOSURE[chemistry]
    ca = p.coord[(p.res_id == 1) & (p.atom_name == atom_a)]
    cb = p.coord[(p.res_id == LENGTH) & (p.atom_name == atom_b)]

    view = py3Dmol.view(width=680, height=460)
    view.addModel(to_pdb(arr[arr.chain_id == tgt_chain]), "pdb")
    view.setStyle({"cartoon": {"color": "0xb9c0c9"}})
    view.addStyle({"resi": HOTSPOTS}, {"stick": {"colorscheme": "orangeCarbon", "radius": 0.3}})

    view.addModel(to_pdb(p), "pdb")
    view.setStyle({"model": -1}, {"cartoon": {"color": "0x3b82f6"},
                                  "stick": {"colorscheme": "cyanCarbon", "radius": 0.15}})
    if len(ca) and len(cb):
        d = float(np.linalg.norm(ca[0] - cb[0]))
        view.addLine({"start": dict(zip("xyz", map(float, ca[0]))),
                      "end": dict(zip("xyz", map(float, cb[0]))),
                      "color": "0x16a34a", "dashed": True, "linewidth": 4})
        print(f"{chemistry}: closure {atom_a}1-{atom_b}{LENGTH} = {d:.3f} A (ideal {ideal})")
    view.setBackgroundColor("0xffffff")
    view.zoomTo({"model": -1})
    view.show()


show("head_tail")

In [ ]:
show("disulfide")

In [ ]:
show("isopeptide")

## 6. Results

Measured on this input set, 16 replicates per chemistry, seed 0, released checkpoint,
one H200. Closed means `|d - ideal| <= 0.3 A`; failures stay in the denominator.

| Chemistry | Closed | Median (A) | Range (A) | Hotspots engaged |
|---|---|---|---|---|
| head-to-tail | **16/16** | 1.318 | 1.277-1.390 | 4 of 4 |
| disulfide | 12/16 | 2.119 | 1.576-2.724 | 4 of 4 |
| isopeptide | **1/16** | 2.136 | 1.478-4.412 | 4 of 4 |

**All three dock.** Every chemistry engages all four hotspots with a comparable interface
(median 114-164 peptide-target atom contacts within 5 A). Binding is not what separates
them — ring closure is.

**Head-to-tail is production-ready.** 16/16 at 1.318 A against the ideal 1.33, with a
0.11 A spread. This matches its behaviour on coordinate-free inputs generally: in a separate
sweep over lengths 6-20 it closed 360/360 at median 1.308 A, with no length dependence.

**Disulfide works but is imprecise.** 12/16 here; over lengths 6-20 it closes 177/360, with
the useful band at 10-16 and near-total failure at 6-7, where the ring cannot physically
accommodate the bridge. The median sits almost exactly on the ideal, so the problem is
spread rather than aim — budget replicates and filter on the measured distance.

**Isopeptide does not work de novo.** 1/16 here, 18/360 in the length sweep. With input
coordinates it is reliable (24/24 at 1.363 A when the crosslinked residues are atomized), so
this is a property of the regime, not of the chemistry — and atomization is unavailable for
a designed chain because atomized tokens need finite input coordinates.

Worth knowing about the failure mode: in the length sweep the model usually closed onto the
wrong atom, placing `LYS.NZ` against the carboxyl **oxygen** `OD1` (median 1.614 A, the
closest contact in 243/360 designs) rather than forming the `NZ`-`CG` amide (2.325 A). A
threshold on the declared pair alone cannot distinguish that from a ring that simply did not
close, which is why the measurement here reports which atoms ended up closest.

## Checklist

- Enumerate the peptide chain explicitly. A length-range placeholder cannot carry a closure
  bond.
- Declare the closure as `_struct_conn` with `conn_type_id = covale`. RCSB `disulf` records
  are not read by the parse preset.
- Leave `expseg` bounds at `-1` on a chain that is not an expandable segment. `0` deletes it.
- `import rfproteina.conditions` before parsing or writing, or the hotspots vanish.
- Validate with `check_annotated_cif.sh` and look for `struct_conn_bond_missing` before
  spending GPU time.
- Fix `C_SEQ` on the residues the chemistry depends on, or the model will design them away.
- Measure closure with a two-sided threshold, measure hotspot engagement separately, and
  keep failures in the denominator.
- For crosslinks, generate extra replicates and filter on the measured distance.